# Thực nghiệm 1 — DIODE zero-shot relative depth

Depth Anything V1, CVPR 2024 Table 2. Native validation: **325 indoor + 446 outdoor = 771 ảnh**.

Dùng checkpoint relative ViT-L; mask cảm biến chính thức; fit scale/shift và chấm trong disparity; không crop hoặc depth cap. Mốc paper: **AbsRel 0.066, δ₁ 0.952**. Đây là protocol tái hiện được ghi rõ; các chi tiết split/clipping/TTA của paper chưa được xác nhận đầy đủ. Xem [hướng dẫn](docs/table2/diode-relative-depth.md).

Chạy notebook từ thư mục gốc repo bằng môi trường `.venv`. Để chạy đủ validation, giữ `SUBSET='all'` và `LIMIT=None`. Cache cho phép tiếp tục hoặc chấm lại mà không suy luận lại toàn bộ.

**Kiểm chứng dữ liệu:** archive chính thức có MD5 khớp; 2.313 file RGB/depth/mask local giống từng byte. Mỗi lần chạy notebook kiểm tra lại SHA256 theo report trước suy luận. Lượt fresh rerun lưu riêng trong `audit-rerun`; không thay đổi ground truth để khớp paper.


In [1]:
from pathlib import Path
import pandas as pd
from evaluate_diode_relative import evaluate_diode

assert Path('data/diode/val').is_dir(), 'Hãy chạy notebook từ repo root'
ENCODER = 'vitl'
SUBSET = 'all'  # all | indoors | outdoor
LIMIT = None  # Số ảnh đầu tiên để chẩn đoán; None chạy đủ subset
FLIP = False

DATA_AUDIT = Path('results/table2/diode/audit/official-archive-verification.json')
CACHE_DIR = Path('.cache/diode-audit/fresh-predictions')
OUTPUT_DIR = Path('results/table2/diode/audit-rerun')
REFRESH_CACHE = False  # True: suy luận lại toàn bộ; False: dùng cache hợp lệ


In [2]:
summary = evaluate_diode(
    encoder=ENCODER, subset=SUBSET, limit=LIMIT, flip=FLIP,
    data_dir=Path('data'), checkpoint_dir=Path('checkpoints'),
    data_audit=DATA_AUDIT, refresh_cache=REFRESH_CACHE,
    cache_dir=CACHE_DIR, output_dir=OUTPUT_DIR,
)


DIODE vitl all:   0%|          | 0/771 [00:00<?, ?it/s]

DIODE vitl all:  17%|█▋        | 129/771 [00:10<00:49, 12.89it/s]

DIODE vitl all:  17%|█▋        | 129/771 [00:20<00:49, 12.89it/s]

DIODE vitl all:  33%|███▎      | 257/771 [00:20<00:40, 12.78it/s]

DIODE vitl all:  50%|████▉     | 385/771 [00:31<00:31, 12.27it/s]

DIODE vitl all:  68%|██████▊   | 522/771 [00:41<00:19, 12.80it/s]

DIODE vitl all:  85%|████████▌ | 659/771 [00:51<00:08, 12.82it/s]

DIODE vitl all: 100%|██████████| 771/771 [01:00<00:00, 12.77it/s]

{
  "created_at": "2026-11-26T15:54:17.164071+00:00",
  "num_images": 771,
  "metrics": {
    "abs_rel": 0.2678724206100358,
    "delta1": 0.7536446312872283
  },
  "valid_pixels": 527773509,
  "nonpositive_predictions": 242811,
  "subset": "all",
  "full_requested_split": true,
  "full_validation_split": true,
  "by_domain": {
    "indoors": {
      "num_images": 325,
      "metrics": {
        "abs_rel": 0.057393578895084556,
        "delta1": 0.9557522894205296
      },
      "valid_pixels": 253024007,
      "nonpositive_predictions": 79411
    },
    "outdoor": {
      "num_images": 446,
      "metrics": {
        "abs_rel": 0.42124825818259,
        "delta1": 0.6063688714367282
      },
      "valid_pixels": 274749502,
      "nonpositive_predictions": 163400
    }
  },
  "paper_table": 2,
  "paper_reference": {
    "abs_rel": 0.066,
    "delta1": 0.952
  },
  "paper_comparison_scope": "Published DIODE aggregate; domain/subset runs are diagnostic",
  "difference_from_paper": {
    

In [3]:
rows = [{'Phạm vi': domain, 'Số ảnh': result['num_images'],
         'AbsRel ↓': result['metrics']['abs_rel'], 'δ₁ ↑': result['metrics']['delta1']}
        for domain, result in summary['by_domain'].items()]
rows.append({'Phạm vi': 'Toàn bộ đã chạy', 'Số ảnh': summary['num_images'],
             'AbsRel ↓': summary['metrics']['abs_rel'], 'δ₁ ↑': summary['metrics']['delta1']})
rows.append({'Phạm vi': 'Paper Table 2 (DIODE)', 'Số ảnh': None,
             'AbsRel ↓': summary['paper_reference']['abs_rel'],
             'δ₁ ↑': summary['paper_reference']['delta1']})
pd.DataFrame(rows)


,Phạm vi,Số ảnh,AbsRel ↓,δ₁ ↑
0,indoors,325.0,0.057394,0.955752
1,outdoor,446.0,0.421248,0.606369
2,Toàn bộ đã chạy,771.0,0.267872,0.753645
3,Paper Table 2 (DIODE),NaN,0.066000,0.952000


Kết quả JSON, per-image CSV và manifest được lưu tại `results/table2/diode/`. Dòng paper chỉ là mốc aggregate DIODE; không dùng làm mốc riêng cho indoor/outdoor hoặc tập có `LIMIT`.

Chưa khớp paper không đồng nghĩa chỉ sai làm tròn: cần đối chiếu protocol. Không bỏ ảnh/pixel lỗi hoặc đổi ngưỡng để ép khớp kết quả.
